# 01 · Bronze : ingestion brute

Copie les CSV de `Files/brut` dans des tables du lakehouse, **sans rien corriger**.
Toutes les colonnes restent en texte : la couche bronze garde une trace fidèle de la source.

**Prérequis :** ce notebook est attaché à un lakehouse par défaut, et les CSV sont dans `Files/brut`.

In [ ]:
CHEMIN_BRUT = "Files/brut"
FORMAT_TABLE = "delta"

In [ ]:
from pyspark.sql import functions as F

COLONNES_REQUISES = {
    "bons_de_travail": ["bon_id", "date_creation", "statut", "categorie", "pavillon_id", "local_id"],
    "locaux": ["local_id", "pavillon_id"],
    "pavillons": ["pavillon_id"],
    "techniciens": ["technicien_id"],
    "priorites": ["priorite"],
}

## Lecture et contrôle

`inferSchema` reste désactivé : deviner les types ferait disparaître les dates et les nombres mal formés
que la couche silver doit justement détecter.

Tous les fichiers sont contrôlés avant la première écriture. Si l'un d'eux est vide ou incomplet, le notebook
échoue et les tables de la dernière exécution réussie restent intactes.

In [ ]:
fichiers = {}
for table, colonnes in COLONNES_REQUISES.items():
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("escape", '"')
        .csv(f"{CHEMIN_BRUT}/{table}.csv")
    )
    manquantes = set(colonnes) - set(df.columns)
    assert not manquantes, f"{table}.csv : colonnes manquantes {sorted(manquantes)}"
    assert df.take(1), f"{table}.csv ne contient aucune ligne"
    fichiers[table] = df

## Écriture

In [ ]:
for table, df in fichiers.items():
    (
        df.withColumn("_date_ingestion", F.current_timestamp())
        .withColumn("_fichier_source", F.input_file_name())
        .write.format(FORMAT_TABLE)
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"bronze_{table}")
    )
    print(f"bronze_{table} : {spark.table(f'bronze_{table}').count()} lignes")